# Monte Carlo Dropout and Deep Ensembles

**Module 06 · AI Safety (ISO/PAS 8800)** · Notebook 3 of 7

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milinpatel07/Autonomous-Driving_AI-Safety-and-Security/blob/main/06_AI_Safety_ISO_PAS_8800/03_mc_dropout_and_deep_ensembles.ipynb)

| Level | Time | Prerequisites | Standards |
|---|---|---|---|
| Intermediate | ~60 min | [02 Uncertainty types](02_uncertainty_types.ipynb); PyTorch basics | ISO/PAS 8800:2024, ISO 21448:2022 |

**Author:** Milin Patel · Hochschule Kempten · MIT Licence

## Learning objectives

- Implement **Monte Carlo (MC) dropout** correctly: dropout active at test time, *every other layer (e.g. BatchNorm) in evaluation mode*.
- Implement a **deep ensemble** and compute the entropy / mutual-information decomposition from both methods.
- Evaluate uncertainty properly: accuracy, NLL and Brier score on a **held-out in-distribution (ID) test set**, and OOD detection with **AUROC** and **FPR at 95 % TPR** on near and far OOD sets.
- Recognise the limits: ReLU networks can stay confident far from the data, so uncertainty is not a guaranteed OOD detector.
- Measure the inference cost honestly (warm-up, repeated timing, batched sampling).

## Introduction

[02](02_uncertainty_types.ipynb) introduced the uncertainty types. This notebook shows how to **estimate** them with two widely used methods:

1. **MC dropout:** a single network, sampled several times with random dropout masks.
2. **Deep ensembles:** several independently trained networks.

Both are standard research baselines for uncertainty estimation in perception; how widely they are used in production vehicle stacks is not publicly documented.

In [ ]:
import time
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import make_moons
from sklearn.metrics import roc_auc_score, roc_curve

rng = np.random.default_rng(42)
torch.manual_seed(42)
torch.set_num_threads(max(1, min(4, torch.get_num_threads())))
device = torch.device("cpu")          # the models are tiny; CPU keeps results reproducible
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})
OI = ["#E69F00", "#56B4E9", "#009E73", "#F0E442", "#0072B2", "#D55E00", "#CC79A7", "#000000"]

def boxplot(ax, data, labels, **kw):
    """Matplotlib >= 3.9 renamed boxplot(labels=...) to tick_labels=...; support both."""
    major, minor = (int(v) for v in matplotlib.__version__.split(".")[:2])
    key = "tick_labels" if (major, minor) >= (3, 9) else "labels"
    return ax.boxplot(data, **{key: labels}, **kw)

print("torch", torch.__version__, "| matplotlib", matplotlib.__version__)

## 1. Data: training set, held-out ID test set, near and far OOD

- **Training:** 300 points of the two-moons problem.
- **ID test:** 500 *new* points from the same distribution. Evaluating "in-distribution" uncertainty on training points would understate it.
- **Near OOD:** a cluster just outside the data, inside the plotted region.
- **Far OOD:** points about ten times further away, to illustrate a known failure mode of ReLU networks.

In [ ]:
X_train, y_train = make_moons(n_samples=300, noise=0.2, random_state=42)
X_test, y_test = make_moons(n_samples=500, noise=0.2, random_state=7)           # held-out ID
X_ood_near = rng.normal(loc=[-1.6, 2.0], scale=0.25, size=(200, 2))              # near OOD
X_ood_far = rng.normal(loc=[-15.0, 15.0], scale=2.0, size=(200, 2))              # far OOD

t = lambda a: torch.as_tensor(a, dtype=torch.float32)

lo = np.minimum(X_train.min(0), X_ood_near.min(0)) - 0.7
hi = np.maximum(X_train.max(0), X_ood_near.max(0)) + 0.7
xx, yy = np.meshgrid(np.linspace(lo[0], hi[0], 120), np.linspace(lo[1], hi[1], 120))
grid = np.c_[xx.ravel(), yy.ravel()]
inside = np.mean(np.all((X_ood_near >= lo) & (X_ood_near <= hi), axis=1))

plt.figure(figsize=(9, 6))
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap="viridis", s=25, edgecolor="black", lw=0.4, label="Training")
plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap="viridis", s=10, alpha=0.35, marker="s", label="ID test")
plt.scatter(X_ood_near[:, 0], X_ood_near[:, 1], color=OI[5], marker="x", s=40, label="Near OOD")
plt.xlim(lo[0], hi[0]); plt.ylim(lo[1], hi[1])
plt.xlabel("Feature 1 (-)"); plt.ylabel("Feature 2 (-)"); plt.title("Training, held-out ID test and near-OOD data")
plt.legend(); plt.tight_layout(); plt.show()
print(f"Train {len(X_train)}, ID test {len(X_test)}, near OOD {len(X_ood_near)} ({inside:.0%} inside the plotted grid), far OOD {len(X_ood_far)}")

## 2. Monte Carlo dropout

### 2.1 Theory

**Gal & Ghahramani (2016):** training a network with dropout *and L2 weight regularisation* can be interpreted as approximate variational inference with a Bernoulli variational distribution over the weights. Keeping dropout active at test time and running $S$ forward passes draws $S$ samples from this approximate posterior:

$$p(y\mid x) \approx \frac1S\sum_{s=1}^S p(y\mid x, \hat\theta_s), \qquad \hat\theta_s = \theta \odot \text{mask}_s .$$

From the $S$ softmax outputs we compute the decomposition of [02](02_uncertainty_types.ipynb): total predictive entropy = expected entropy (aleatoric) + mutual information (epistemic). For regression, the predictive variance is $\tau^{-1} + \frac1S\sum_s \hat y_s^2 - \bar y^2$, where only the sample-variance part is epistemic.

### 2.2 The BatchNorm pitfall

A common shortcut is to call `model.train()` before sampling. That also puts **BatchNorm** into training mode: it normalises with the statistics of the current test batch and *updates its running statistics* with test data. Predictions then depend on what else is in the batch, and the model changes as it is used. The correct way is to put the model in `eval()` mode and switch on **only** the dropout modules.

In [ ]:
class MCDropoutClassifier(nn.Module):
    """MLP with BatchNorm and dropout (BatchNorm included on purpose to show the pitfall)."""
    def __init__(self, input_dim=2, hidden=64, num_classes=2, p=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden), nn.BatchNorm1d(hidden), nn.ReLU(), nn.Dropout(p),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(p),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(p),
            nn.Linear(hidden, num_classes))
    def forward(self, x):
        return self.net(x)

def enable_mc_dropout(model):
    """Evaluation mode everywhere, except dropout layers, which keep sampling masks."""
    model.eval()
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.train()

@torch.no_grad()
def mc_dropout_probs(model, x, n_samples=30):
    """Returns P with shape (S, N, C). Samples are drawn in one batched pass for speed."""
    enable_mc_dropout(model)
    n = x.shape[0]
    logits = model(x.repeat(n_samples, 1))              # each copy gets independent dropout masks
    model.eval()
    return F.softmax(logits, dim=-1).reshape(n_samples, n, -1)

def decompose(P):
    """P: (S, N, C) tensor or array -> mean probs, total entropy, aleatoric (expected entropy), epistemic (MI)."""
    P = torch.as_tensor(P)
    mean = P.mean(0)
    total = -(mean * torch.log(mean.clamp_min(1e-12))).sum(-1)
    aleatoric = -(P * torch.log(P.clamp_min(1e-12))).sum(-1).mean(0)
    return mean.numpy(), total.numpy(), aleatoric.numpy(), (total - aleatoric).clamp_min(0).numpy()

def train_model(model, X, y, epochs=300, lr=0.01, weight_decay=1e-4, batch_size=64, seed=0):
    g = torch.Generator().manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    Xt, yt = t(X), torch.as_tensor(y)
    losses = []
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(len(Xt), generator=g)
        for i in range(0, len(Xt), batch_size):
            idx = perm[i:i + batch_size]
            opt.zero_grad(); loss = F.cross_entropy(model(Xt[idx]), yt[idx]); loss.backward(); opt.step()
        losses.append(loss.item())
    model.eval()
    return losses

torch.manual_seed(0)
mc_model = MCDropoutClassifier()
t0 = time.perf_counter()
losses = train_model(mc_model, X_train, y_train, epochs=150, seed=0)
print(f"MC-dropout model trained in {time.perf_counter() - t0:.1f} s, final mini-batch loss {losses[-1]:.3f}")

In [ ]:
# Demonstrate the pitfall: model.train() during inference changes BatchNorm running statistics
import copy
bn = mc_model.net[1]
saved_state = copy.deepcopy(mc_model.state_dict())

before = bn.running_mean.clone()
mc_model.train()
with torch.no_grad():
    _ = mc_model(t(X_ood_near))                       # "inference" in train mode on an OOD batch
changed = (bn.running_mean - before).abs().max().item()
mc_model.load_state_dict(saved_state)                 # undo the damage for the rest of the notebook

before = bn.running_mean.clone()
_ = mc_dropout_probs(mc_model, t(X_ood_near), n_samples=10)
unchanged = (bn.running_mean - before).abs().max().item()

# Batch dependence: identical input and identical dropout seed, different batch-mates
x0 = t(X_test[:1])
with torch.no_grad():
    enable_mc_dropout(mc_model); torch.manual_seed(1); a = F.softmax(mc_model(torch.cat([x0, t(X_test[1:64])])), -1)[0, 1].item()
    mc_model.train(); torch.manual_seed(1); b = F.softmax(mc_model(torch.cat([x0, t(X_ood_near[:63])])), -1)[0, 1].item()
mc_model.load_state_dict(saved_state); mc_model.eval()
print(f"model.train() at inference changed BatchNorm running_mean by up to {changed:.3f}")
print(f"enable_mc_dropout() changed it by {unchanged:.1e} (should be 0)")
print(f"Same input, P(class 1): {a:.3f} with dropout-only sampling vs {b:.3f} in train mode with an OOD batch")

## 3. Deep ensembles

**Lakshminarayanan et al. (2017):** train $M$ networks from different random initialisations (and data order). Their average is the predictive distribution, and their **disagreement** is the epistemic signal. The members here have no dropout and no BatchNorm, so ordinary `eval()` mode is correct.

$$p(y\mid x) \approx \frac1M\sum_{m=1}^M p(y\mid x,\theta_m)$$

Typical sizes in the literature are $M = 5$ to $10$. Training and memory cost grow linearly with $M$; inference can be batched or parallelised.

In [ ]:
class StandardClassifier(nn.Module):
    def __init__(self, input_dim=2, hidden=64, num_classes=2):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(input_dim, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU(),
                                 nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, num_classes))
    def forward(self, x):
        return self.net(x)

class DeepEnsemble:
    def __init__(self, num_models=5, seed=100):
        self.models = []
        for i in range(num_models):
            torch.manual_seed(seed + i)        # explicit, documented seed per member -> different initialisations
            self.models.append(StandardClassifier())
        self.seed = seed
    def fit(self, X, y, **kw):
        for i, m in enumerate(self.models):
            train_model(m, X, y, seed=self.seed + i, **kw)
        return self
    @torch.no_grad()
    def probs(self, x):
        return torch.stack([F.softmax(m.eval()(x), dim=-1) for m in self.models])    # (M, N, C)

t0 = time.perf_counter()
ensemble = DeepEnsemble(num_models=5).fit(X_train, y_train, epochs=150, weight_decay=0.0)
print(f"Ensemble of {len(ensemble.models)} trained in {time.perf_counter() - t0:.1f} s")

In [ ]:
S_MC = 30
torch.manual_seed(3)
methods = {
    "MC dropout (S=30)": lambda x: mc_dropout_probs(mc_model, t(x), n_samples=S_MC),
    "Deep ensemble (M=5)": lambda x: ensemble.probs(t(x)),
}
maps = {name: decompose(f(grid)) for name, f in methods.items()}

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for row, (name, (mean, total, ale, epi)) in enumerate(maps.items()):
    for col, (Z, title, cmap) in enumerate([(mean[:, 1], "Mean P(class 1)", "viridis"),
                                            (ale, "Aleatoric: expected entropy (nats)", "Blues"),
                                            (epi, "Epistemic: mutual information (nats)", "Reds")]):
        ax = axes[row, col]
        cs = ax.contourf(xx, yy, Z.reshape(xx.shape), levels=20, cmap=cmap)
        plt.colorbar(cs, ax=ax)
        ax.scatter(X_train[:, 0], X_train[:, 1], c="white", s=8, edgecolor="black", lw=0.3)
        ax.scatter(X_ood_near[:, 0], X_ood_near[:, 1], c="black", marker="x", s=15)
        ax.set_title(f"{name}\n{title}", fontsize=11)
        ax.set_xlabel("Feature 1 (-)"); ax.set_ylabel("Feature 2 (-)")
plt.tight_layout(); plt.show()

for name, (mean, total, ale, epi) in maps.items():
    near_bound = np.abs(mean[:, 1] - 0.5) < 0.15
    print(f"{name}: mean MI on grid points near the decision boundary {epi[near_bound].mean():.3f}, "
          f"elsewhere {epi[~near_bound].mean():.3f}; mean aleatoric near boundary {ale[near_bound].mean():.3f}, "
          f"elsewhere {ale[~near_bound].mean():.3f} nats")

## 4. Evaluation on held-out ID data and OOD data

Two different questions:

1. **Predictive quality on ID data:** accuracy, negative log-likelihood (NLL) and Brier score. NLL and Brier are *proper scoring rules*: they reward calibrated probabilities, not just correct argmax decisions.
2. **OOD detection:** can an uncertainty score separate ID test inputs from OOD inputs? We report the threshold-free **AUROC** (0.5 = chance) and the **FPR at 95 % TPR**, i.e. the fraction of ID inputs falsely flagged when 95 % of OOD inputs are caught.

In [ ]:
def nll_brier_acc(mean_probs, y):
    p_true = mean_probs[np.arange(len(y)), y]
    onehot = np.eye(mean_probs.shape[1])[y]
    return (-np.log(np.clip(p_true, 1e-12, 1)).mean(),
            np.mean(np.sum((mean_probs - onehot) ** 2, axis=1)),
            np.mean(mean_probs.argmax(1) == y))

def ood_metrics(score_id, score_ood):
    labels = np.r_[np.zeros(len(score_id)), np.ones(len(score_ood))]
    scores = np.r_[score_id, score_ood]
    fpr, tpr, _ = roc_curve(labels, scores)
    return roc_auc_score(labels, scores), fpr[np.searchsorted(tpr, 0.95)]

# single deterministic network (first ensemble member) as a baseline
single = lambda x: ensemble.probs(t(x))[:1]
all_methods = {"Single network": single, **methods}

torch.manual_seed(4)
results, scores = [], {}
for name, f in all_methods.items():
    out = {split: decompose(f(X)) for split, X in [("id", X_test), ("near", X_ood_near), ("far", X_ood_far)]}
    nll, brier, acc = nll_brier_acc(out["id"][0], y_test)
    row = dict(method=name, acc=acc, nll=nll, brier=brier)
    for split in ("near", "far"):
        for key, idx in [("H", 1), ("MI", 3)]:
            if name == "Single network" and key == "MI":
                continue                                     # MI is zero for a single network
            auc, fpr95 = ood_metrics(out["id"][idx], out[split][idx])
            row[f"AUROC {split} ({key})"] = auc
            row[f"FPR95 {split} ({key})"] = fpr95
        row[f"max-p {split}"] = out[split][0].max(1).mean()
    results.append(row); scores[name] = out

import pandas as pd
res = pd.DataFrame(results).set_index("method")
pd.set_option("display.width", 200)
print(res.round(3).T.to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, name in zip(axes[:2], methods):
    out = scores[name]
    boxplot(ax, [out["id"][3], out["near"][3], out["far"][3]], ["ID test", "Near OOD", "Far OOD"], patch_artist=True)
    ax.set_yscale("symlog", linthresh=1e-3)
    ax.set_ylabel("Mutual information (nats)"); ax.set_title(f"{name}: epistemic score by data set")
ax = axes[2]
for name, c in zip(all_methods, [OI[7], OI[0], OI[4]]):
    out = scores[name]
    lab = np.r_[np.zeros(len(out["id"][1])), np.ones(len(out["near"][1]))]
    fpr, tpr, _ = roc_curve(lab, np.r_[out["id"][1], out["near"][1]])
    ax.plot(fpr, tpr, color=c, lw=2, label=f"{name} (AUROC {roc_auc_score(lab, np.r_[out['id'][1], out['near'][1]]):.2f})")
ax.plot([0, 1], [0, 1], "k:", lw=1)
ax.set_xlabel("False positive rate (ID flagged)"); ax.set_ylabel("True positive rate (OOD flagged)")
ax.set_title("ROC: ID test vs near OOD, score = total entropy"); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

# Conclusions derived from the numbers above
best_near = res["AUROC near (H)"].idxmax()
print(f"Best near-OOD detector by total entropy: {best_near} (AUROC {res.loc[best_near, 'AUROC near (H)']:.3f}).")
for name in methods:
    r = res.loc[name]
    print(f"{name}: near-OOD AUROC with MI {r['AUROC near (MI)']:.3f} vs total entropy {r['AUROC near (H)']:.3f}; "
          f"far-OOD AUROC with MI {r['AUROC far (MI)']:.3f}; mean max-probability on far OOD {r['max-p far']:.3f}.")
conf_far = res["max-p far"].min()
if conf_far > 0.9:
    print(f"Every method still assigns a mean max-probability of at least {conf_far:.2f} to far-OOD points: "
          "confident predictions far from the data, as described by Hein et al. (2019).")
else:
    print(f"Lowest mean max-probability on far OOD: {conf_far:.2f}.")

**Reading the results:**
- Check whether the ID metrics (accuracy, NLL, Brier) differ much between methods; on such a simple problem they often do not.
- OOD detection quality depends strongly on *which* OOD data are used. A method that separates near OOD well may still be confident on far OOD, because ReLU networks extrapolate linearly and become more confident far from the data (Hein et al., 2019).
- Uncertainty-based OOD detection therefore needs its own verification per ODD slice and cannot be argued from first principles. In a safety case it is one monitor among several (input-distribution checks, cross-sensor plausibility, ODD monitoring); see [04_SOTIF/03](../04_SOTIF_ISO21448/03_ood_monitoring_as_safety_measure.ipynb) and [02_Failures/03](../02_Failures_and_Robustness/03_out_of_distribution_detection.ipynb).

## 5. Computational cost

Timing uses `time.perf_counter()`, a warm-up call, and the median of repeated runs. MC dropout is timed both as a Python loop of $S$ passes and as **one batched pass** over $S$ copies of the input.

In [ ]:
def time_it(fn, reps=15):
    fn()                                             # warm-up
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

x_bench = t(rng.standard_normal((1000, 2)))

@torch.no_grad()
def mc_loop(n):
    enable_mc_dropout(mc_model)
    out = torch.stack([F.softmax(mc_model(x_bench), -1) for _ in range(n)])
    mc_model.eval(); return out

@torch.no_grad()
def single_forward():
    return F.softmax(ensemble.models[0](x_bench), -1)

timings = {"Single network": time_it(single_forward)}
for n in (10, 30):
    timings[f"MC dropout loop (S={n})"] = time_it(lambda: mc_loop(n))
    timings[f"MC dropout batched (S={n})"] = time_it(lambda: mc_dropout_probs(mc_model, x_bench, n))
timings["Ensemble (M=5)"] = time_it(lambda: ensemble.probs(x_bench))

base = timings["Single network"]
fig, ax = plt.subplots(figsize=(11, 4.5))
names = list(timings); vals = [timings[k] * 1e3 for k in names]
bars = ax.bar(names, vals, color=[OI[7]] + [OI[0], OI[1]] * 2 + [OI[4]], edgecolor="black")
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.2f} ms\n({v / (base * 1e3):.0f}x)", ha="center", va="bottom", fontsize=8)
ax.set_ylabel("Median inference time for 1,000 inputs (ms)"); ax.set_title("Measured inference cost on this CPU")
plt.xticks(rotation=20, ha="right"); plt.tight_layout(); plt.show()

n_params = lambda m: sum(p.numel() for p in m.parameters())
print(f"Parameters: MC-dropout model {n_params(mc_model):,}; ensemble {sum(n_params(m) for m in ensemble.models):,} "
      f"({len(ensemble.models)} x {n_params(ensemble.models[0]):,})")
for k, v in timings.items():
    print(f"{k:28s} {v*1e3:7.2f} ms  ({v / base:5.1f}x single network)")
print(f"Batching the S=30 MC samples is {timings['MC dropout loop (S=30)'] / timings['MC dropout batched (S=30)']:.1f}x "
      "faster than the Python loop here; on a vehicle accelerator, memory and latency budgets decide which form is feasible.")

## 6. Other uncertainty quantification methods (brief overview)

### 6.1 Bayesian neural networks
A distribution over weights $p(\theta\mid\mathcal D)$ instead of a point estimate. Exact inference is intractable; variational inference, MCMC and the **Laplace approximation** (Daxberger et al., 2021) are practical approximations. Priors and approximations strongly affect the result.

### 6.2 Evidential deep learning
The network predicts the parameters of a **Dirichlet** (higher-order) distribution over class probabilities in a single forward pass (Sensoy et al., 2018). It is fast, but its epistemic-uncertainty estimates are debated: the learned "epistemic" measures need not vanish with infinite data (Bengs et al., 2022), and deep evidential regression has been criticised as well (Meinert et al., 2023).

### 6.3 Comparison summary

In [ ]:
comparison = pd.DataFrame({
    "Method": ["Single network", "MC dropout", "Deep ensemble", "Bayesian NN (VI / Laplace)", "Evidential DL"],
    "Training cost": ["1x", "1x", "Mx", "1-3x", "1x"],
    "Inference cost": ["1x", "Sx (batchable)", "Mx (parallelisable)", "Sx", "1x"],
    "Memory": ["1x", "1x", "Mx", "1-2x", "1x"],
    "Aleatoric": ["softmax only*", "yes*", "yes*", "yes*", "yes (Dirichlet)"],
    "Epistemic": ["no", "yes (approx.)", "yes", "yes (approx.)", "debated"],
    "Implementation": ["easy", "easy", "easy", "medium-hard", "medium"],
})
print(comparison.to_string(index=False))
print("\n*via the likelihood head (softmax entropy, or a predicted variance for regression).")
print("S = number of MC samples, M = number of ensemble members.")
print("\nMeasured in this notebook:")
timing_key = {"Single network": "Single network", "MC dropout (S=30)": "MC dropout batched (S=30)",
              "Deep ensemble (M=5)": "Ensemble (M=5)"}
for name in all_methods:
    r = res.loc[name]
    print(f"  {name:22s} ID NLL {r['nll']:.3f} | near-OOD AUROC (entropy) {r['AUROC near (H)']:.3f} | "
          f"time for 1,000 inputs {timings[timing_key[name]] * 1e3:.2f} ms")

## Key takeaways

- **MC dropout:** sample dropout masks at test time with all other layers in evaluation mode; `model.train()` corrupts BatchNorm statistics and makes predictions batch-dependent.
- **Deep ensembles:** independently initialised members; disagreement is the epistemic signal. Cost scales with $M$.
- Report the **entropy / mutual-information decomposition**, not only the standard deviation of one class probability.
- Evaluate on a **held-out ID test set** with proper scoring rules (NLL, Brier) and evaluate OOD detection with **AUROC and FPR@95TPR** on clearly defined OOD sets.
- Uncertainty from MC dropout or ensembles is **not a guaranteed OOD detector**: far from the data, ReLU networks can remain confident. Any monitor built on it must be verified like any other safety mechanism (ISO/PAS 8800, ISO 21448).

## Self-assessment

1. Why should `model.train()` not be used to enable MC dropout in a network with BatchNorm?
2. How is the mutual information computed from $S$ MC-dropout samples?
3. What does AUROC = 0.5 mean for an OOD detector? What does FPR@95TPR = 0.4 mean?
4. Why is evaluating "ID uncertainty" on training points misleading?
5. Give one argument for and one against deep ensembles on an embedded vehicle computer.
6. Why can a deep ensemble still be confident on inputs very far from the training data?

<details>
<summary>Answers</summary>

1. It switches BatchNorm to batch statistics and updates its running statistics with test data, so predictions depend on the other inputs in the batch and the model drifts during use. Only dropout layers should be in training mode.
2. With $P_s$ the softmax vector of sample $s$: total = $H(\frac1S\sum_s P_s)$, aleatoric = $\frac1S\sum_s H(P_s)$, MI = total − aleatoric.
3. AUROC 0.5: the score ranks ID and OOD inputs no better than chance. FPR@95TPR 0.4: to catch 95 % of OOD inputs, 40 % of normal ID inputs would also be flagged, which is unusable as a runtime trigger.
4. The model has fitted those points, so its uncertainty there is lower than on new data from the same distribution; the comparison with OOD data becomes optimistic.
5. For: usually the most reliable of the simple methods, easy to implement, members can run in parallel. Against: M times memory and compute, which may exceed the accelerator's budget; members may also share systematic errors from the same data.
6. Piecewise-linear (ReLU) networks extrapolate linearly; far from the data, logits grow and the softmax saturates. If all members extrapolate in a similar direction, they agree and the mutual information is small (Hein et al., 2019).
</details>

## References

- Gal, Y., Ghahramani, Z. (2016). Dropout as a Bayesian approximation: representing model uncertainty in deep learning. ICML 2016. arXiv:1506.02142
- Lakshminarayanan, B., Pritzel, A., Blundell, C. (2017). Simple and scalable predictive uncertainty estimation using deep ensembles. NeurIPS 2017. arXiv:1612.01474
- Depeweg, S. et al. (2018). Decomposition of uncertainty in Bayesian deep learning. ICML 2018. arXiv:1710.07283
- Hein, M., Andriushchenko, M., Bitterwolf, J. (2019). Why ReLU networks yield high-confidence predictions far away from the training data and how to mitigate the problem. CVPR 2019. arXiv:1812.05720
- Ovadia, Y. et al. (2019). Can you trust your model's uncertainty? Evaluating predictive uncertainty under dataset shift. NeurIPS 2019. arXiv:1906.02530
- Hendrycks, D., Gimpel, K. (2017). A baseline for detecting misclassified and out-of-distribution examples in neural networks. ICLR 2017. arXiv:1610.02136
- Sensoy, M., Kaplan, L., Kandemir, M. (2018). Evidential deep learning to quantify classification uncertainty. NeurIPS 2018. arXiv:1806.01768
- Bengs, V., Hüllermeier, E., Waegeman, W. (2022). Pitfalls of epistemic uncertainty quantification through loss minimisation. NeurIPS 2022. arXiv:2203.06102
- Meinert, N., Gawlikowski, J., Lavin, A. (2023). The unreasonable effectiveness of deep evidential regression. AAAI 2023. arXiv:2205.10060
- Daxberger, E. et al. (2021). Laplace Redux — effortless Bayesian deep learning. NeurIPS 2021. arXiv:2106.14806
- Patel, M., Jung, R. (2026). Uncertainty Evaluation to Support Safety of the Intended Functionality Analysis for Identifying Performance Insufficiencies in ML-Based LiDAR Object Detection. VEHITS 2026. https://doi.org/10.5220/0014967200004030

---
**Navigation:** ← [02 Uncertainty types](02_uncertainty_types.ipynb) · [Module index](README.md) · [04 Calibration](04_calibration.ipynb) →

### Further practice

**Try this:** change the dropout rate (0.1, 0.3, 0.5), the number of MC samples (10, 100) and the ensemble size (3, 10), and watch the near- and far-OOD AUROC. Combine both methods (an ensemble of MC-dropout networks).

The author's companion repository [AI-Perception-Safety-Workshop](https://github.com/milinpatel07/AI-Perception-Safety-Workshop) has an MC-dropout exercise on images.